# Week 60, Day 6: Forge: Week 60 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w060/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
K, d = 4, 3
print("1.", (K - 1) + K * d + K * d * (d + 1) // 2)
print("2.", round(np.exp(-0.5 * 1) / (np.exp(-0.5 * 1) + np.exp(-0.5 * 3)), 4))
print("3.", 3)

### Exercise: Part A: a Poisson mixture

Complete the E step (responsibilities from pi times Poisson probabilities, computed in logs) and the M step (weights and rates) of poisson_mixture, fit it to the counts, and score the inferred states against the truth.

In [ ]:
import numpy as np
from scipy.special import gammaln, logsumexp
gA = np.random.default_rng(606)
state = gA.random(1000) < 0.35                       # True in the up state
counts = gA.poisson(np.where(state, 6.0, 0.4))

def poisson_mixture(x, K, iters=200):
    lam = np.quantile(x, np.linspace(0.2, 0.8, K)) + 0.5
    pi = np.full(K, 1 / K); hist = []
    for _ in range(iters):
        logp = np.log(pi) + x[:, None] * np.log(lam) - lam - gammaln(x[:, None] + 1)
        hist.append(logsumexp(logp, axis=1).sum())
        R = ...                                      # E step
        Nk = R.sum(0)
        pi = ...                                     # M step
        lam = ...
        if len(hist) > 1 and hist[-1] - hist[-2] < 1e-9:
            break
    return pi, lam, R, hist

pi_hat, lam_hat, R_pm, ll_pm = poisson_mixture(counts, 2)
state_hat = R_pm[:, int(np.argmax(lam_hat))] > 0.5
state_acc = np.mean(state_hat == state)
print(f"weights {np.round(pi_hat, 3)}, rates {np.round(lam_hat, 2)} spikes per bin; {len(ll_pm)} iterations")
print(f"hidden state recovered on {state_acc:.3f} of bins")

In [ ]:
import numpy as np
def test_loglik():
    "the log-likelihood never decreases"
    assert all(b >= a - 1e-9 for a, b in zip(ll_pm, ll_pm[1:]))
def test_params():
    "the fitted weight and rates are close to the truth (0.35 up, rates 0.4 and 6 spikes per bin)"
    lo, hi = sorted(lam_hat)
    assert abs(hi - 6) < 0.5 and abs(lo - 0.4) < 0.2 and abs(pi_hat[int(np.argmax(lam_hat))] - 0.35) < 0.05
def test_states():
    "assigning bins by responsibility recovers the hidden state on more than 95% of bins"
    assert state_acc > 0.95

test_loglik()
test_params()
test_states()
print("All checks passed")

### Exercise: Part B: shrinking the components

Write soft_kmeans(X, C0, sigma), running 100 EM iterations with responsibilities proportional to exp(-squared distance / 2 sigma squared) and centres updated as responsibility-weighted means. For each sigma, record the mean of each point's largest responsibility and the agreement of the hard assignments with k-means from the same start.

In [ ]:
gB = np.random.default_rng(6061)
Xb = np.vstack([gB.normal(c, 0.7, (80, 2)) for c in [[0, 0], [3, 1], [1, 3.5]]])
C0 = Xb[gB.choice(len(Xb), 3, replace=False)]

def hard_kmeans(X, C, iters=100):
    C = C.copy()
    for _ in range(iters):
        lab = np.argmin(((X[:, None] - C[None]) ** 2).sum(-1), 1)
        C = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(len(C))])
    return lab

def soft_kmeans(X, C0, sigma, iters=100):
    C = C0.copy()
    for _ in range(iters):
        D2 = ((X[:, None] - C[None]) ** 2).sum(-1)
        logits = -D2 / (2 * sigma ** 2)
        R = ...
        C = ...
    return R

lab_km = hard_kmeans(Xb, C0)
sigmas = [2.0, 1.0, 0.5, 0.2, 0.05]
hardness, match = [], []
for s_ in sigmas:
    R = soft_kmeans(Xb, C0, s_)
    hardness.append(R.max(1).mean()); match.append(np.mean(R.argmax(1) == lab_km))
for s_, h, m in zip(sigmas, hardness, match):
    print(f"sigma {s_:4.2f}: mean largest responsibility {h:.3f}, agreement with k-means {m:.3f}")

In [ ]:
import numpy as np
def test_hard():
    "responsibilities harden as sigma shrinks: mean max responsibility above 0.99 at the smallest sigma, below 0.8 at the largest"
    assert hardness[-1] > 0.99 and hardness[0] < 0.8
def test_match():
    "at the smallest sigma, soft k-means assignments equal those of k-means"
    assert match[-1] == 1.0

test_hard()
test_match()
print("All checks passed")